In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Admin\\Downloads\\kidney-diseases-classification\\research'

In [3]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True) 
class DataIngestionConfig: 
    root_dir: Path 
    source_URL: str 
    local_data_file: Path
    unzip_dir: Path

In [4]:
# Notebook mein test
import sys
print(sys.path)  # cnnClassifier path dikhna chahiye

import cnnClassifier
print("✅ Package found!")


['C:\\Users\\Admin\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip', 'C:\\Users\\Admin\\AppData\\Local\\Python\\pythoncore-3.14-64\\DLLs', 'C:\\Users\\Admin\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib', 'C:\\Users\\Admin\\AppData\\Local\\Python\\pythoncore-3.14-64', 'c:\\Users\\Admin\\Downloads\\kidney-diseases-classification\\kidneyv', '', 'c:\\Users\\Admin\\Downloads\\kidney-diseases-classification\\kidneyv\\Lib\\site-packages', 'C:\\Users\\Admin\\Downloads\\kidney-diseases-classification\\src', 'c:\\Users\\Admin\\Downloads\\kidney-diseases-classification\\kidneyv\\Lib\\site-packages\\win32', 'c:\\Users\\Admin\\Downloads\\kidney-diseases-classification\\kidneyv\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\Admin\\Downloads\\kidney-diseases-classification\\kidneyv\\Lib\\site-packages\\Pythonwin']
✅ Package found!


In [5]:
from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
print("Config:", CONFIG_FILE_PATH)
print("Params:", PARAMS_FILE_PATH)
print("✅ Constants LOADED!")


Config: C:\Users\Admin\Downloads\kidney-diseases-classification\config\config.yaml
Params: C:\Users\Admin\Downloads\kidney-diseases-classification\params.yaml
✅ Constants LOADED!


In [6]:
from cnnClassifier.constants import *
print("Config path:", CONFIG_FILE_PATH)
print("Config exists:", CONFIG_FILE_PATH.exists())


Config path: C:\Users\Admin\Downloads\kidney-diseases-classification\config\config.yaml
Config exists: True


In [7]:
# Notebook test cell
from cnnClassifier.constants import CONFIG_FILE_PATH
print(CONFIG_FILE_PATH)  # config/config.yaml


C:\Users\Admin\Downloads\kidney-diseases-classification\config\config.yaml


In [8]:
from cnnClassifier.constants import *  
from cnnClassifier.utils.common import read_yaml, create_directories



In [9]:
from pathlib import Path
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

print("Config exists:", CONFIG_FILE_PATH.exists())
print("Params exists:", PARAMS_FILE_PATH.exists())


Config exists: True
Params exists: True


In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [11]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [12]:
import os
import gdown
import zipfile
from cnnClassifier import logger

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            logger.info(f"Downloading data from Google Drive...")
            # Extract file ID from Google Drive URL
            file_id = self.config.source_URL.split("/")[-2]
            
            gdown.download(
                id=file_id,
                output=str(self.config.local_data_file),
                quiet=False,
                resume=True
            )
            logger.info(f"File downloaded to: {self.config.local_data_file}")
        else:
            logger.info(f"File already exists at: {self.config.local_data_file}")

    def extract_zip_file(self):
        """
        Extracts the zip file into the data directory
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Zip file extracted to: {unzip_path}")


In [13]:
import os
from pathlib import Path

# Paths
config_path = Path(r"C:\Users\Admin\Downloads\kidney-diseases-classification\config\config.yaml")
params_path = Path(r"C:\Users\Admin\Downloads\kidney-diseases-classification\params.yaml")

# 1. Config YAML Content
config_content = """artifacts_root: artifacts

data_ingestion:
  root_dir: artifacts/data_ingestion
  source_URL: "https://drive.google.com/file/d/1yLBJ1RQufaHNnr6HUNmx51Rjaupw_g_s/view?usp=drive_link"
  local_data_file: artifacts/data_ingestion/data.zip
  unzip_dir: artifacts/data_ingestion
"""

# 2. Params YAML Content
params_content = """data_ingestion:
  unzip_dir: artifacts/data_ingestion
"""

# Delete old corrupt files and write clean new ones in UTF-8
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config_content)
    
with open(params_path, "w", encoding="utf-8") as f:
    f.write(params_content)

print("✅ Files recreated successfully with clean UTF-8 encoding!")


✅ Files recreated successfully with clean UTF-8 encoding!


In [14]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e 


[2026-03-16 16:38:09,017]: yaml file: C:\Users\Admin\Downloads\kidney-diseases-classification\config\config.yaml loaded successfully:
[2026-03-16 16:38:09,021]: yaml file: C:\Users\Admin\Downloads\kidney-diseases-classification\params.yaml loaded successfully:
[2026-03-16 16:38:09,023]: created directory at: artifacts:
[2026-03-16 16:38:09,025]: created directory at: artifacts/data_ingestion:
[2026-03-16 16:38:09,026]: Downloading data from Google Drive...:


Downloading...
From (original): https://drive.google.com/uc?id=1yLBJ1RQufaHNnr6HUNmx51Rjaupw_g_s
From (redirected): https://drive.google.com/uc?id=1yLBJ1RQufaHNnr6HUNmx51Rjaupw_g_s&confirm=t&uuid=3d9a42ce-bbca-423d-ba53-2d2b57ecd99c
To: c:\Users\Admin\Downloads\kidney-diseases-classification\research\artifacts\data_ingestion\data.zip
100%|██████████| 1.63G/1.63G [00:28<00:00, 56.9MB/s]

[2026-03-16 16:38:40,540]: File downloaded to: artifacts/data_ingestion/data.zip:


[2026-03-16 16:38:54,775]: Zip file extracted to: artifacts/data_ingestion:
